# 服飾資料庫：Colab ↔ TiDB Cloud
流程：上傳 SQL → 連線 TiDB → 建表/匯入/加 image_url → 驗證 → 匯出 `products.json` → 推到 GitHub Pages

## 1. 安裝套件

In [ ]:
!pip -q install pymysql

## 2. 上傳 SQL 檔
一次選取 `01_schema.sql`、`02_data.sql`、`03_image_url.sql`、`04_verify.sql`

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

## 3. 連線 TiDB Cloud
到 TiDB Cloud → Connect → 取得 Host / User / Password（Port 4000，需 TLS）。密碼用輸入框，不要寫進筆記本。

In [ ]:
import pymysql, getpass
HOST = input("TiDB Host (例 gateway01.ap-northeast-1.prod.aws.tidbcloud.com): ").strip()
USER = input("User (例 xxxxxxxx.root): ").strip()
PWD  = getpass.getpass("Password: ")

def connect(db=None):
    return pymysql.connect(host=HOST, port=4000, user=USER, password=PWD, database=db,
                           charset="utf8mb4", ssl_ca="/etc/ssl/certs/ca-certificates.crt",
                           ssl_verify_cert=True, ssl_verify_identity=True, autocommit=True)

conn = connect()
with conn.cursor() as cur:
    cur.execute("SELECT VERSION()"); print("連線成功：", cur.fetchone())

## 4. 依序執行 SQL（建表 → 匯入 → image_url）

In [ ]:
import re
def run_sql_file(name):
    text = open(name, encoding="utf-8").read()
    stmts = [s.strip() for s in re.split(r";\s*\n", text) if s.strip()]
    ok = 0
    with conn.cursor() as cur:
        for s in stmts:
            s = "\n".join(l for l in s.splitlines() if not l.strip().startswith("--")).strip().rstrip(";")
            if not s: continue
            try:
                cur.execute(s); ok += 1
            except pymysql.err.MySQLError as e:
                if e.args[0] in (1060, 1050):   # 欄位/表已存在：重跑時略過
                    print("略過（已存在）：", e.args[1]); continue
                raise
    print(f"{name}: 執行 {ok} 個陳述式")

for f in ["01_schema.sql", "02_data.sql", "03_image_url.sql"]:
    run_sql_file(f)

## 5. 驗證（截圖放作業：資料庫截圖）

In [ ]:
import pandas as pd
conn = connect("clothing_store")
for t in ["categories","brands","clothing","customers","orders","order_items","product_images"]:
    print(t, pd.read_sql(f"SELECT COUNT(*) n FROM {t}", conn).iloc[0,0])
display(pd.read_sql("""SELECT c.product_id, c.product_name, ca.category_name, b.brand_name, c.price, c.stock_qty, c.image_url, i.match_level
FROM clothing c JOIN categories ca USING(category_id) JOIN brands b USING(brand_id)
LEFT JOIN product_images i ON i.product_id=c.product_id ORDER BY c.product_id LIMIT 12""", conn))
display(pd.read_sql("""SELECT ca.category_name, COUNT(*) 款數, SUM(i.match_level='exact') exact, SUM(i.match_level='close') close_, SUM(i.match_level='sample') sample_
FROM clothing c JOIN categories ca USING(category_id) JOIN product_images i ON i.product_id=c.product_id GROUP BY ca.category_name""", conn))

## 6. 匯出 `products.json`（dashboard 讀這個檔）

In [ ]:
import json, os
df = pd.read_sql("""SELECT c.product_id, c.product_name, c.category_id, ca.category_name, c.brand_id, b.brand_name, b.country,
  c.color, c.size, c.price, c.stock_qty, c.listed_date, c.image_url, i.match_level
FROM clothing c JOIN categories ca USING(category_id) JOIN brands b USING(brand_id)
LEFT JOIN product_images i ON i.product_id=c.product_id AND i.image_type='main' ORDER BY c.product_id""", conn)
df["price"] = df["price"].astype(float); df["listed_date"] = df["listed_date"].astype(str)
records = df.to_dict("records")
os.makedirs("docs/data", exist_ok=True)
with open("docs/data/products.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=1)
print(len(records), "筆已匯出")

## 7. 推送到 GitHub（更新 `docs/data/products.json`，Pages 約 1 分鐘後生效）
需要 GitHub Personal Access Token（Fine-grained，對該 repo 開 Contents: Read and write）。

In [ ]:
import base64, requests
REPO = "你的帳號/clothing-search"      # ← 改成你的 repo
BRANCH = "main"
TOKEN = getpass.getpass("GitHub Token: ")
url = f"https://api.github.com/repos/{REPO}/contents/docs/data/products.json"
H = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"}
sha = requests.get(url, headers=H, params={"ref": BRANCH}).json().get("sha")
body = {"message": "update products.json from TiDB", "branch": BRANCH,
        "content": base64.b64encode(open("docs/data/products.json","rb").read()).decode()}
if sha: body["sha"] = sha
r = requests.put(url, headers=H, json=body)
print(r.status_code, r.json().get("content", {}).get("html_url", r.text[:200]))